In [33]:
import pandas as pd
from matplotlib.figure import Figure
import matplotlib.pyplot as plt

import panel as pn

# ipywidgets - The Matplotlib pane allows displaying Matplotlib figures inside a Panel app. This includes figures created by Seaborn, Pandas .plot, Plotnine and any other plotting library building on top of Matplotlib.

pn.extension('matplotlib')

In [34]:
def load_data():
    data = pd.read_excel('./../3-data-analysis/data/titanic/Titanic.xls')
    return data

# load_data()

In [35]:
df = load_data()

In [36]:
def make_boxplots(df):
    fig, axes = plt.subplots(nrows=2, 
                             ncols=2,
                             figsize=(8, 4),
                             tight_layout=True)
    df[['age']].plot.box(ax=axes[0,0], 
                         fontsize=7,
                         vert=False,
                         color='red')
    axes[0,0].set_title('Age', fontsize=8)
    axes[0,0].tick_params(labelsize=5)


    import seaborn as sns
    g = sns.violinplot(ax=axes[0,1],
                   data=df,
                   x='pclass',
                   y='age',
                   hue='sex')
    # put the legend outside the plot
    g.legend(loc='center left', bbox_to_anchor=(1, 0.5), fontsize=7)
    axes[0,1].set_title('Age distribution by class', fontsize=8)
    axes[0,1].set_xlabel('Passenger Class', fontsize=7)
    axes[0,1].set_ylabel('Age', fontsize=7)
    axes[0,1].tick_params(labelsize=5)

    df['age'].plot(kind='hist',
                   ax=axes[1,0],
                   alpha=0.1,
                   bins=20,
                   color='green')
    axes[1,0].set_xlabel('Age', fontsize=7)
    axes[1,0].set_ylabel('Count', fontsize=7)
    axes[1,0].set_title('Age distribution', fontsize=8)
    axes[1,0].tick_params(labelsize=5)

    g = df.pivot_table(index='sex',values='survived', columns='pclass', aggfunc='sum')\
        .plot(kind='bar', ax=axes[1,1])
    g.legend(loc='center left', bbox_to_anchor=(1, 0.5), fontsize=7)
    axes[1,1].set_title('Survivors by gender and class', fontsize=8)
    axes[1,1].set_ylabel('Number of survivors', fontsize=7)
    axes[1,1].tick_params(labelsize=5)

    return fig

In [38]:
df_panel = pn.widgets.DataFrame(df, width=800, height=400)
df_plots = pn.pane.Matplotlib(make_boxplots(df))

data_visualization_panel = pn.layout.Column(
    df_plots,
    df_panel
)

In [ ]:
def update_indicator(event):
    text_widgetvalue = filter_text_intput.value
    sex_widget_values = filter_sex.value # list of selected values
    passenger_class_widget_values = filter_passenger_class.value # list of selected values
    survived_widget_values = filter_survived.value # list of selected values

    # build filter based on widget values - start with all True        
    filter = pd.Series([True] * len(df))

    # filter on name
    if text_widgetvalue:
        filter = filter & df['name'].str.contains(text_widgetvalue)
    
    # filter on sex
    if sex_widget_values:
        filter = filter & df['sex'].isin(sex_widget_values)
    
    # filter on passenger class
    if passenger_class_widget_values:
        passenger_class_widget_values_as_int = [int(x) for x in passenger_class_widget_values]
        filter = filter & df['pclass'].isin(passenger_class_widget_values_as_int)  
        
    # filter on survived
    if survived_widget_values:
        survived_widget_values_as_int = [int(x) for x in survived_widget_values]
        filter = filter & df['survived'].isin(survived_widget_values_as_int)

    # filter_text_intput.value = survived_widget_values_as_int
    
    # update the DataFrame panel    
    df_panel.value = df[filter] 
    
    # update the boxplot
    df_plots.object = make_boxplots(df[filter])

In [14]:
# Text filter on the passenger name
filter_text_intput = pn.widgets.TextInput(name='Filter Name', value='')

# Button to apply the text filter
filter_button = pn.widgets.Button(name='Apply Filter', button_type='primary')
pn.bind(update_indicator, filter_button, watch=True)

# MultiSelect to filter on sex
filter_sex = pn.widgets.MultiSelect(name='Sex', 
                                    options=['male', 'female'], 
                                    value=['male', 'female'])
pn.bind(update_indicator, filter_sex, watch=True)

# MultiSelect to filter on passenger class
filter_passenger_class = pn.widgets.MultiSelect(name='Passenger Class', 
                                                options=['1', '2', '3'],
                                                value=['1', '2', '3'])
pn.bind(update_indicator, filter_passenger_class, watch=True)

# MultiSelect to filter if passenger survived
filter_survived = pn.widgets.MultiSelect(name='Survived', 
                                         options=[0, 1],
                                         value=[0, 1])
pn.bind(update_indicator, filter_survived, watch=True)

# Panel to hold the filter widgets
filter_panel = pn.layout.Column(
    pn.layout.Row(
        filter_text_intput,
        filter_button),
    pn.layout.Row(
        filter_sex,
        filter_passenger_class,
        filter_survived)
)

In [15]:
pn.layout.Column(
    filter_panel,
    data_visualization_panel
).servable()

Column
    [0] Row
        [0] TextInput(name='Filter')
        [1] Button(button_type='primary', name='Apply Filter')
    [1] DataFrame(value=      pclass  survived    ...)